# Task 2 - Connect Four con Minimax

## Que vas a ver aqui
1. Una clase `Connect4` para manejar el tablero.
2. Funciones para detectar victorias y estados terminales.
3. Un agente Minimax puro (sin poda) para elegir jugadas.

## Regla de puntuacion usada
- Gana IA: valor alto positivo.
- Gana jugador: valor alto negativo.
- Empate: 0.

## Nota importante
La profundidad se limita a `d = 3` o `d = 4` porque Minimax puro crece muy rapido en tiempo de calculo.

In [10]:
# imports y  constantes 
import math
import copy
import random


ROWS = 6
COLS = 7
EMPTY = 0
PLAYER = 1
AI = 2

## Task 2.1 Class Connect4

Esta clase manejará:
- el estado del tablero
- los movimientos válidos
- la colocación de fichas
- la detección de victoria
- la verificación de empate o estado terminal

Se usará una matriz de 6 filas por 7 columnas, donde:
- `0` representa una casilla vacía
- `1` representa una ficha del jugador
- `2` representa una ficha de la IA

In [11]:
class Connect4:
    def __init__(self):
        self.board = [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]

    def clone(self):
        new_game = Connect4()
        new_game.board = copy.deepcopy(self.board)
        return new_game

    def print_board(self):
        for row in self.board:
            print(row)
        print("0 1 2 3 4 5 6")
        print()

    def actions(self):
        """
        Devuelve una lista con las columnas válidas donde aún se puede jugar.
        """
        valid_moves = []
        for col in range(COLS):
            if self.board[0][col] == EMPTY:
                valid_moves.append(col)
        return valid_moves

    def drop_piece(self, col, piece):
        """
        Coloca una ficha en la columna indicada.
        La ficha cae hasta la posición más baja disponible.
        Devuelve True si se pudo colocar, False si la jugada no es válida.
        """
        if col < 0 or col >= COLS or self.board[0][col] != EMPTY:
            return False

        for row in range(ROWS - 1, -1, -1):
            if self.board[row][col] == EMPTY:
                self.board[row][col] = piece
                return True

        return False

    def is_full(self):
        """
        Devuelve True si ya no hay movimientos válidos.
        """
        return len(self.actions()) == 0

    def check_winner(self, piece):
        """
        Revisa si la ficha indicada tiene 4 en línea.
        """

        # Horizontal
        for row in range(ROWS):
            for col in range(COLS - 3):
                if (
                    self.board[row][col] == piece and
                    self.board[row][col + 1] == piece and
                    self.board[row][col + 2] == piece and
                    self.board[row][col + 3] == piece
                ):
                    return True

        # Vertical
        for row in range(ROWS - 3):
            for col in range(COLS):
                if (
                    self.board[row][col] == piece and
                    self.board[row + 1][col] == piece and
                    self.board[row + 2][col] == piece and
                    self.board[row + 3][col] == piece
                ):
                    return True

        # Diagonal descendente hacia la derecha
        for row in range(ROWS - 3):
            for col in range(COLS - 3):
                if (
                    self.board[row][col] == piece and
                    self.board[row + 1][col + 1] == piece and
                    self.board[row + 2][col + 2] == piece and
                    self.board[row + 3][col + 3] == piece
                ):
                    return True

        # Diagonal ascendente hacia la derecha
        for row in range(3, ROWS):
            for col in range(COLS - 3):
                if (
                    self.board[row][col] == piece and
                    self.board[row - 1][col + 1] == piece and
                    self.board[row - 2][col + 2] == piece and
                    self.board[row - 3][col + 3] == piece
                ):
                    return True

        return False

    def is_terminal(self):
        """
        Un estado terminal ocurre si:
        - gana PLAYER
        - gana AI
        - o el tablero está lleno
        """
        return self.check_winner(PLAYER) or self.check_winner(AI) or self.is_full()

## Prueba básica del tablero

En esta parte se prueba:
- crear un tablero vacío
- colocar fichas
- mostrar el tablero
- verificar movimientos válidos

In [12]:
game = Connect4()
game.print_board()

game.drop_piece(3, PLAYER)
game.drop_piece(3, AI)
game.drop_piece(2, PLAYER)
game.drop_piece(4, AI)

game.print_board()
print("Movimientos válidos:", game.actions())
print("¿Es terminal?", game.is_terminal())

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
0 1 2 3 4 5 6

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 1, 1, 2, 0, 0]
0 1 2 3 4 5 6

Movimientos válidos: [0, 1, 2, 3, 4, 5, 6]
¿Es terminal? False


## Task 2.1: Agente Minimax (Puro)

En este bloque:
- `evaluate_basic` da una evaluacion simple.
- `minimax` explora jugadas de forma recursiva.
- `get_best_move` prueba cada columna valida y elige la mejor para la IA.

Primero buscamos que funcione correctamente y sea claro; luego se puede mejorar con una heuristica mas avanzada o poda alfa-beta.

In [13]:
minimax_nodes_visited = 0


def evaluate_basic(board):
    """Heurística básica: solo detecta victorias directas."""
    if board.check_winner(AI):
        return 1000
    elif board.check_winner(PLAYER):
        return -1000
    else:
        return 0


def minimax(board, depth, maximizing_player):
    """Calcula el valor minimax del tablero actual."""
    global minimax_nodes_visited
    minimax_nodes_visited += 1

    is_term = board.is_terminal()

    # Caso base: fin de profundidad o estado terminal.
    if depth == 0 or is_term:
        if is_term:
            if board.check_winner(AI):
                return 100000000
            elif board.check_winner(PLAYER):
                return -100000000
            else:
                return 0
        return evaluate_basic(board)

    valid_locations = board.actions()
    if not valid_locations:
        return evaluate_basic(board)

    if maximizing_player:
        value = -math.inf
        for col in valid_locations:
            b_copy = board.clone()
            b_copy.drop_piece(col, AI)
            new_score = minimax(b_copy, depth - 1, False)
            if new_score > value:
                value = new_score
        return value

    value = math.inf
    for col in valid_locations:
        b_copy = board.clone()
        b_copy.drop_piece(col, PLAYER)
        new_score = minimax(b_copy, depth - 1, True)
        if new_score < value:
            value = new_score
    return value


def get_best_move(board, depth):
    """Devuelve la mejor columna para la IA."""
    global minimax_nodes_visited
    minimax_nodes_visited = 0

    valid_locations = board.actions()
    best_score = -math.inf
    best_col = valid_locations[0] if valid_locations else None

    for col in valid_locations:
        b_copy = board.clone()
        b_copy.drop_piece(col, AI)
        score = minimax(b_copy, depth - 1, False)

        if score > best_score:
            best_score = score
            best_col = col

    return best_col


print("Mejor columna (d=3):", get_best_move(game, 3))
print("Nodos visitados:", minimax_nodes_visited)

Mejor columna (d=3): 0
Nodos visitados: 399
